In [5]:
import pandas as pd
from pycaret.regression import setup, compare_models, tune_model, predict_model, finalize_model, pull, evaluate_model

In [6]:
data = pd.read_csv(r"D:\Sem 4\PBL\data\processed\data_proses_bulanan.csv")
data.head()

,Order Date,Quantity,Sales,Profit,Discount,Quantity_Lag_1,Quantity_Lag_2,Quantity_Rolling_Mean_3,Sales_Lag_1,Profit_Lag_1,Month,Year,Sub-Category
0,2014-04-30,0,0.00,0.00,0.0,0.0,2.0,1.333333,0.00,0.0,4,2014,Copiers
1,2014-05-31,3,719.98,135.00,0.2,0.0,0.0,0.666667,0.00,0.0,5,2014,Copiers
2,2014-06-30,1,559.99,175.00,0.2,3.0,0.0,1.000000,719.98,135.0,6,2014,Copiers
3,2014-07-31,0,0.00,0.00,0.0,1.0,3.0,1.333333,559.99,175.0,7,2014,Copiers
4,2014-08-31,9,3549.94,949.99,0.2,0.0,1.0,1.333333,0.00,0.0,8,2014,Copiers


In [9]:

# =========================================================================
# 1. MENYIAPKAN DATA DAN WADAH DOKUMENTASI BULANAN
# =========================================================================
# Wadah untuk dokumentasi top 3 baseline model bulanan
tabel_dokumentasi_monthly = []

# SIlakan sesuaikan 'data' dengan nama DataFrame bulananmu (misal: df_monthly_master)
# Kita ambil daftar sub-kategori unik (seperti Machine, Copier, dll.)
sub_categories_monthly = data['Sub-Category'].unique()

print("Memulai Pemodelan Otomatis (BULANAN) dengan Tracking MLflow...")

# Looping otomatis untuk setiap sub-kategori bulanan
for sub_cat in sub_categories_monthly:
    print(f"\n=======================================================")
    print(f"[PROSES BULANAN] Menghitung Model untuk Sub-Kategori: {sub_cat}")
    print(f"=======================================================")
    
    # Filter data bulanan per sub-kategori
    df_sub = data[data['Sub-Category'] == sub_cat].copy()
    
    # Proteksi data sedikit: Karena rentang bulanan datanya cenderung lebih sedikit,
    # kita set minimal 6 baris (6 bulan) agar PyCaret masih bisa melakukan split data.
    if len(df_sub) < 6:
        print(f"[LEWAT] Sub-kategori {sub_cat} dilewati karena data terlalu sedikit ({len(df_sub)} baris).")
        continue
    
    # Drop kolom data bocor / teks mentah
    # CATATAN: Jika nama kolom tanggalmu di data bulanan berubah (misal jadi 'Month' atau 'Order Month'),
    # silakan tambahkan atau ganti nama 'Order Date' di bawah ini sesuaikan dengan datamu.
    cols_to_drop = ['Order Date', 'Sales', 'Profit', 'Sub-Category']
    X_data = df_sub.drop(columns=[col for col in cols_to_drop if col in df_sub.columns], errors='ignore')
    
    # 2. Setup PyCaret + Otomatis Log ke MLflow (Skala Bulanan)
    grid = setup(
        data=X_data, 
        target='Quantity', 
        session_id=42,
        log_experiment=True,                       # AKTIFKAN LOG OTOMATIS MLflow
        experiment_name=f"Monthly_{sub_cat}",      # PREFIX 'Monthly_' AGAR TERPISAH DI MLFLOW UI
        log_plots=True,                            # Otomatis mengirimkan plot evaluasi ke MLflow
        verbose=False,                             
        html=False
    )
    
    # 3. Cari Top 3 Model Terbaik berdasarkan MAE (Exclude CatBoost demi keamanan bug clone)
    top3_models = compare_models(
    n_select=3,
    include=['lr', 'lasso', 'ridge', 'en', 'huber'], 
    sort='MAE', 
    verbose=False
)
    print(f"[SUKSES] Top 3 Model Bulanan untuk {sub_cat} telah otomatis tercatat di MLflow.")

    # Ambil papan skor metrik menggunakan pull()
    leaderboard = pull() 
    top3_metrics = leaderboard.head(3)
    
    # Masukkan baris demi baris hasil baseline ke dalam list dokumentasi bulanan
    for urutan, (nama_model_baseline, baris_metrik) in enumerate(top3_metrics.iterrows(), 1):
        tabel_dokumentasi_monthly.append({
            'Sub-Category': sub_cat,
            'Rank': f"Top {urutan}",
            'Model Name': nama_model_baseline,
            'MAE': baris_metrik['MAE'],
            'RMSE': baris_metrik['RMSE'],
            'R2': baris_metrik['R2'],
            'MAPE': baris_metrik['MAPE']
        })

    # 4. Ambil model peringkat 1 sebagai default awal untuk proses selanjutnya
    model_terpilih = top3_models[0] 
    nama_model = type(model_terpilih).__name__ 
    
    # =========================================================================
    # BLOK PROTEKSI: TUNING MODEL TOP 1 BULANAN
    # =========================================================================
    try:
        print(f"-> Mencoba melakukan tuning pada model: {nama_model}...")
        tuned_model = tune_model(model_terpilih, optimize='MAE', verbose=False)
        
        # Jika sukses, variabel model_terpilih diperbarui menjadi versi tuning
        model_terpilih = tuned_model
        print(f"[SUKSES] Model {nama_model} BERHASIL di-tuning!")
        
    except Exception as e:
        print(f"[PERINGATAN] Model {nama_model} tidak mendukung otomatis tuning (Error Clone).")
        print(f"-> Solusi Otomatis: Tetap menggunakan model {nama_model} versi ASLI (Tanpa Tuning).")

    # =========================================================================
    # PROSES AKHIR PER SUB-KATEGORI: PREDICT -> FINALIZE
    # =========================================================================
    # a. Predict Model (Evaluasi pada data Holdout / Test set bulanan)
    predict_model(model_terpilih, verbose=False)
    print(f"[SUKSES] Prediksi & Holdout Metrics Bulanan untuk {sub_cat} telah tercatat di MLflow.")
    
    # b. Evaluasi Model (Tampilkan plot evaluasi di notebook dan kirim ke MLflow)
    # Langkah ini akan otomatis mengirimkan plot evaluasi ke MLflow
    evaluate_model(model_terpilih)
    print(f"[SUKSES] Evaluasi Model untuk {sub_cat} telah tercatat di MLflow.")

    # c. Finalize Model (Latihan ulang dengan 100% gabungan data Train + Test)
    final_model = finalize_model(model_terpilih)
    print(f"[SUKSES] Model terbaik untuk {sub_cat} telah difinalisasi dan siap dipakai forecasting.")

# =========================================================================
# KELUAR DARI LOOPING: PROSES AKHIR DOKUMENTASI CSV BULANAN
# =========================================================================
print("\n=======================================================")
print("[PROSES SELESAI] Membuat DataFrame Bulanan dan Menyimpan CSV...")
print("=======================================================")

# Mengubah list kumpulan data tadi menjadi DataFrame Pandas yang rapi
df_hasil_monthly = pd.DataFrame(tabel_dokumentasi_monthly)

# Menyimpan DataFrame menjadi file CSV khusus data bulanan agar tidak menimpa data mingguan
df_hasil_monthly.to_csv('dokumentasi_baseline_models_monthly.csv', index=False)

print("✅ File 'dokumentasi_baseline_models_monthly.csv' berhasil dibuat!")
print("👉 Silakan cek dashboard MLflow UI kamu untuk melihat folder 'Monthly_...'.")
print("=======================================================")

Memulai Pemodelan Otomatis (BULANAN) dengan Tracking MLflow...

[PROSES BULANAN] Menghitung Model untuk Sub-Kategori: Copiers
[SUKSES] Top 3 Model Bulanan untuk Copiers telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: LinearRegression...
[SUKSES] Model LinearRegression BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics Bulanan untuk Copiers telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Copiers telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Copiers telah difinalisasi dan siap dipakai forecasting.

[PROSES BULANAN] Menghitung Model untuk Sub-Kategori: Machines
[SUKSES] Top 3 Model Bulanan untuk Machines telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: Lasso...
[SUKSES] Model Lasso BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics Bulanan untuk Machines telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Machines telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Machines telah difinalisasi dan siap dipakai forecasting.

[PROSES SELESAI] Membuat DataFrame Bulanan dan Menyimpan CSV...
✅ File 'dokumentasi_baseline_models_monthly.csv' berhasil dibuat!
👉 Silakan cek dashboard MLflow UI kamu untuk melihat folder 'Monthly_...'.


In [ ]:
# =========================================================================
# 1. MENYIAPKAN DATA DAN WADAH DOKUMENTASI GLOBAL BULANAN
# =========================================================================
# Wadah untuk dokumentasi top 3 baseline model global
tabel_dokumentasi_monthly = []

print("Memulai Pemodelan Otomatis (GLOBAL BULANAN) dengan Tracking MLflow...")
print("Menggabungkan seluruh sub-kategori menjadi satu kesatuan data...")

# PERUBAHAN UTAMA: Tidak ada looping lagi. Kita langsung pakai seluruh data.
df_global = data.copy()

# Proteksi baris data global
print(f"-> Total baris data global bulanan yang digunakan: {len(df_global)} baris.")

# PERUBAHAN KRUSIAL: Hapus 'Sub-Category' dari daftar cols_to_drop!
# Kita wajib menyisakan 'Sub-Category' agar model bisa membedakan setiap produk.
cols_to_drop = ['Order Date', 'Sales', 'Profit']
X_data_global = df_global.drop(columns=[col for col in cols_to_drop if col in df_global.columns], errors='ignore')

# =========================================================================
# 2. SETUP PYCARET GLOBAL + OTOMATIS LOG KE MLFLOW
# =========================================================================
# Kita tambahkan parameter categorical_features agar PyCaret mengenali kolom teks nama produk
grid_global = setup(
    data=X_data_global, 
    target='Quantity', 
    session_id=42,
    categorical_features=['Sub-Category'],     # MEMBERITAHU PYCARET BAHWA INI FITUR PRODUK
    log_experiment=True,                       # AKTIFKAN LOG OTOMATIS MLflow
    experiment_name="Global_Monthly_Model",    # NAMA EKSPERIMEN BARU YANG RAPI DI MLFLOW
    log_plots=True,                            # Otomatis mengirimkan plot evaluasi ke MLflow
    verbose=False,                             
    html=False
)

# =========================================================================
# 3. CARI TOP 3 BASELINE MODEL TERBAIK (Sesuai daftar model linear pilihanmu)
# =========================================================================
top3_models = compare_models(
    n_select=3,
    exclude=['catboost'],  # Exclude CatBoost demi keamanan bug clone
    sort='MAE', 
    verbose=False
)
print("[SUKSES] Top 3 Model Global telah otomatis tercatat di MLflow.")

# Ambil papan skor metrik menggunakan pull()
leaderboard = pull() 
top3_metrics = leaderboard.head(3)

# Masukkan baris demi baris hasil baseline ke dalam list dokumentasi
for urutan, (nama_model_baseline, baris_metrik) in enumerate(top3_metrics.iterrows(), 1):
    tabel_dokumentasi_monthly.append({
        'Sub-Category': 'Global (All Products)', # Ditulis Global karena mencakup semua produk
        'Rank': f"Top {urutan}",
        'Model Name': nama_model_baseline,
        'MAE': baris_metrik['MAE'],
        'RMSE': baris_metrik['RMSE'],
        'R2': baris_metrik['R2'],
        'MAPE': baris_metrik['MAPE']
    })

# =========================================================================
# 4. AMBIL MODEL PERINGKAT 1 SEBAGAI DEFAULT UNTUK PROSES TUNING
# =========================================================================
model_terpilih = top3_models[0] 
nama_model = type(model_terpilih).__name__ 

# =========================================================================
# BLOK PROTEKSI: TUNING MODEL TOP 1 GLOBAL
# =========================================================================
try:
    print(f"-> Mencoba melakukan tuning pada model global: {nama_model}...")
    tuned_model = tune_model(model_terpilih, optimize='MAE', verbose=False)
    
    # Jika sukses, variabel model_terpilih diperbarui menjadi versi tuning
    model_terpilih = tuned_model
    print(f"[SUKSES] Model {nama_model} BERHASIL di-tuning!")
    
except Exception as e:
    print(f"[PERINGATAN] Model {nama_model} tidak mendukung otomatis tuning.")
    print(f"-> Solusi Otomatis: Tetap menggunakan model {nama_model} versi ASLI (Tanpa Tuning).")

# =========================================================================
# PROSES AKHIR: PREDICT -> EVALUATE -> FINALIZE
# =========================================================================
# a. Predict Model (Evaluasi pada data Holdout / Test set global)
predict_model(model_terpilih, verbose=False)
print(f"[SUKSES] Prediksi & Holdout Metrics Global telah tercatat di MLflow.")

# b. Evaluasi Model (Menampilkan plot evaluasi interaktif di notebook)
evaluate_model(model_terpilih)
print(f"[SUKSES] Evaluasi Model Global telah tercatat di MLflow.")

# c. Finalize Model (Latihan ulang model terpilih dengan 100% data global)
final_model = finalize_model(model_terpilih)
print(f"[SUKSES] Model terbaik Global telah difinalisasi dan siap dipakai forecasting.")

# =========================================================================
# PROSES AKHIR DOKUMENTASI CSV BULANAN GLOBAL
# =========================================================================
print("\n=======================================================")
print("[PROSES SELESAI] Membuat DataFrame Global dan Menyimpan CSV...")
print("=======================================================")

# Mengubah list kumpulan data tadi menjadi DataFrame Pandas yang rapi
df_hasil_monthly = pd.DataFrame(tabel_dokumentasi_monthly)

# Menyimpan DataFrame menjadi file CSV khusus data bulanan global
df_hasil_monthly.to_csv('dokumentasi_baseline_models_monthly.csv', index=False)

print("✅ File 'dokumentasi_baseline_models_monthly.csv' berhasil dibuat!")
print("👉 Silakan cek dashboard MLflow UI kamu untuk melihat folder 'Global_Monthly_Model'.")
print("=======================================================")

Memulai Pemodelan Otomatis (GLOBAL BULANAN) dengan Tracking MLflow...
Menggabungkan seluruh sub-kategori menjadi satu kesatuan data...
-> Total baris data global bulanan yang digunakan: 89 baris.
[SUKSES] Top 3 Model Global telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model global: ExtraTreesRegressor...
[SUKSES] Model ExtraTreesRegressor BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics Global telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model Global telah tercatat di MLflow.
[SUKSES] Model terbaik Global telah difinalisasi dan siap dipakai forecasting.

[PROSES SELESAI] Membuat DataFrame Global dan Menyimpan CSV...
✅ File 'dokumentasi_baseline_models_monthly.csv' berhasil dibuat!
👉 Silakan cek dashboard MLflow UI kamu untuk melihat folder 'Global_Monthly_Model'.
